# Build a 3D FPS Game, Step by Step
### Lesson Guide — Python + the *ursina* engine

We build the game in this order:

1. An empty window  
2. Ground + sky  
3. A player who can walk, look and jump  
4. Walls (a little map)  
5. A crosshair  
6. A gun  
7. Shooting bullets — our first **class**  
8. A shooting **sound**  
9. Swapping the boxy gun for a real 3D model  
10. Tidying it all into a clean, organized project

## How to use this notebook

> A 3D game opens its own window and runs until you close it — that can't happen inside a notebook cell. So **don't run the code cells here.** They exist only so you can read and **copy** them.

**The loop for each step:**
1. Read the explanation.
2. Copy that step's code into a file, e.g. `game.py`, in your `fps_3d` folder.
3. In a terminal: `conda activate pygame_env` then `python game.py`.
4. Play it, press **ESC** to quit, then come back for the next step.

Each step is a **complete, runnable game** — just replace the whole `game.py` with the new step's code each time. Lines marked `# NEW` are what changed since the previous step.

**You'll need:** the `pygame_env` environment with `ursina` installed (`pip install ursina`), and for steps 8–9, the files `shoot.wav` and `blaster.glb` (+ the `Textures/` folder) in the same folder. They're already in your `fps_3d` project.

*(The WSL lines at the top of each program switch on software rendering so the window isn't black, and let it run even with no sound card. They do nothing on a normal computer.)*

## Step 1 — An empty window

Every ursina game has the same three-part skeleton:

- `app = Ursina()` — start the engine and open a window.
- `window.*` — set the title and size.
- `app.run()` — start the game loop (runs until you close the window).

**What you'll see:** a small, mostly empty window. That's success — the engine works!

**Ask the student:** what do you think `app.run()` does? (Answer: it starts an endless loop that redraws the screen ~60 times a second.)

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)
window.fullscreen = False
window.borderless = False

app.run()


## Step 2 — A ground and a sky

An **Entity** is anything that appears in the 3D world. We add two: a flat **ground** (`model='plane'`) and a **sky** (`Sky()`).

- `texture='white_cube'` + `texture_scale` gives the ground a tiled, grid look.
- `collider='box'` makes it solid, so the player can stand on it later.

**Try with the student:** change `color.green` to another colour, or make `scale` bigger.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

# NEW: the floor you'll walk on
ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')

Sky()   # NEW: a bright sky so the world isn't empty

app.run()


## Step 3 — The player (walk, look, jump)

This is where ursina shines. `FirstPersonController` is a ready-made class that gives us **mouse-look + WASD walking + gravity + jumping** all at once.

- `speed` = walking speed. `mouse_sensitivity` = how fast looking feels (smaller = calmer).
- `player.cursor.enabled = False` hides its default pink aiming dot.

### The big teaching point: **inheritance**
*We wrote zero lines of walking or jumping code* — it all lives inside `FirstPersonController`. Later, our own `Player` class will say `class Player(FirstPersonController)`, meaning "a Player **is a** FirstPersonController, and gets all its abilities for free, plus whatever we add."

**Show the student:** in a code editor, Ctrl+Click `FirstPersonController` to jump into ursina's own file. Movement is in its `update()` method; jumping in its `input()`.

**Controls now:** W/A/S/D move, mouse looks, Space jumps.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

# NEW: import the ready-made player controller
from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

# NEW: walk (WASD) + look (mouse) + jump (space), all built in
player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False   # hide the default pink dot

app.run()


## Step 4 — Build a map with walls

A game needs places to walk around. We keep a **list of `(x, z)` spots** and use a `for` loop to place a block at each one. `collider='box'` makes them solid.

**This is the most fun part to edit with your student** — add, remove, or move the numbers in `block_spots` to design your own level. Great way to teach lists and loops.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

# NEW: a block/pillar at each (x, z) spot -- edit this list to design your map!
block_spots = [
    (5, 5), (5, 8), (5, 11),
    (-6, 3), (-6, 6), (-9, 6),
    (0, 12), (3, 12), (-3, 12),
]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

app.run()


## Step 5 — A crosshair

To aim we want a `+` in the middle of the screen. Things attached to **`camera.ui`** are drawn **flat on the screen** (like a sticker), not out in the 3D world. We make one parent and give it two thin white bars — one wide, one tall — which together make a `+`.

**Concept for the student:** *world space* (3D things like blocks) vs *screen space* (flat UI like the crosshair).

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

block_spots = [(5, 5), (5, 8), (5, 11), (-6, 3), (-6, 6), (-9, 6),
               (0, 12), (3, 12), (-3, 12)]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

# NEW: a '+' crosshair drawn flat on the screen
crosshair = Entity(parent=camera.ui)
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.03, 0.004))
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.004, 0.03))

app.run()


## Step 6 — A gun (built from boxes)

Let's hold a gun. We parent it to the **`camera`** (not `camera.ui`), so it lives in the 3D world and follows wherever you look. We build the gun out of three cubes — body, barrel, grip — all **children** of one `gun` entity, so they move together as one piece.

**Try:** change the gun's `position` `(right, up, forward)` to move it on screen.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

block_spots = [(5, 5), (5, 8), (5, 11), (-6, 3), (-6, 6), (-9, 6),
               (0, 12), (3, 12), (-3, 12)]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

crosshair = Entity(parent=camera.ui)
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.03, 0.004))
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.004, 0.03))

# NEW: a gun made of cubes, stuck to the camera at the bottom-middle
gun = Entity(parent=camera, position=(0.35, -0.28, 0.6))
Entity(parent=gun, model='cube', color=color.gray, scale=(0.12, 0.12, 0.45))
Entity(parent=gun, model='cube', color=color.dark_gray,
       scale=(0.05, 0.05, 0.35), position=(0, 0.02, 0.35))
Entity(parent=gun, model='cube', color=color.brown,
       scale=(0.08, 0.22, 0.1), position=(0, -0.15, -0.15))

app.run()


## Step 7 — Shooting!  (our first class)

Now the fun part: **bullets** that fly forward and disappear. This is the perfect moment to introduce a **class**.

### What the `Bullet` class teaches
- **A class bundles look + behaviour together.** `Bullet` *is an* `Entity` (inherits from it) and adds its own data (`direction`, `life`) and behaviour (`update`).
- **ursina calls each entity's `update()` automatically, every frame.** So every bullet moves *itself* — we never need a list of bullets.
- `self.position += self.direction * 40 * time.dt` means **"move 40 units per second."** The `time.dt` (seconds since the last frame) keeps the speed the same on a fast or slow computer.

### The `input(key)` function
ursina automatically calls a function named `input` whenever a key or mouse button is pressed. On a left click we create a new `Bullet`, starting just in front of the camera (`camera.forward * 1.5`) and flying the way we're looking (`direction=camera.forward`).

*(That `input` runs without us calling it — it's an ursina naming convention. Later we'll move this into the `Player` class to make it less "magic.")*

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

block_spots = [(5, 5), (5, 8), (5, 11), (-6, 3), (-6, 6), (-9, 6),
               (0, 12), (3, 12), (-3, 12)]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

crosshair = Entity(parent=camera.ui)
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.03, 0.004))
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.004, 0.03))

gun = Entity(parent=camera, position=(0.35, -0.28, 0.6))
Entity(parent=gun, model='cube', color=color.gray, scale=(0.12, 0.12, 0.45))
Entity(parent=gun, model='cube', color=color.dark_gray,
       scale=(0.05, 0.05, 0.35), position=(0, 0.02, 0.35))
Entity(parent=gun, model='cube', color=color.brown,
       scale=(0.08, 0.22, 0.1), position=(0, -0.15, -0.15))


# NEW: a Bullet class -- it moves and deletes ITSELF every frame
class Bullet(Entity):
    def __init__(self, position, direction):
        super().__init__(model='sphere', color=color.yellow, scale=0.3,
                         position=position, collider='sphere')
        self.direction = direction
        self.life = 3.0

    def update(self):   # ursina runs this every frame, for THIS bullet
        self.position += self.direction * 40 * time.dt
        self.life -= time.dt
        if self.life <= 0:
            destroy(self)


# NEW: ursina runs this on every key/click
def input(key):
    if key == 'left mouse down':
        Bullet(position=camera.world_position + camera.forward * 1.5,
               direction=camera.forward)
    if key == 'escape':
        application.quit()

app.run()


## Step 8 — A shooting sound

Let's add a *pew* when you shoot. The golden rule: **load the sound once** (reading a file is slow), then just **`.play()`** it on each shot.

- `shoot_sound = Audio('shoot.wav', volume=0.5, autoplay=False)` — load it once. `autoplay=False` means "don't play yet."
- `shoot_sound.play()` inside `input` — play on each click.

> Put **`shoot.wav`** in the same folder as your `game.py` (it's already in `fps_3d`). On WSL you won't *hear* it (no sound card), but it plays on Windows. The game runs fine either way.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

block_spots = [(5, 5), (5, 8), (5, 11), (-6, 3), (-6, 6), (-9, 6),
               (0, 12), (3, 12), (-3, 12)]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

crosshair = Entity(parent=camera.ui)
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.03, 0.004))
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.004, 0.03))

gun = Entity(parent=camera, position=(0.35, -0.28, 0.6))
Entity(parent=gun, model='cube', color=color.gray, scale=(0.12, 0.12, 0.45))
Entity(parent=gun, model='cube', color=color.dark_gray,
       scale=(0.05, 0.05, 0.35), position=(0, 0.02, 0.35))
Entity(parent=gun, model='cube', color=color.brown,
       scale=(0.08, 0.22, 0.1), position=(0, -0.15, -0.15))

shoot_sound = Audio('shoot.wav', volume=0.5, autoplay=False)   # NEW: load once


class Bullet(Entity):
    def __init__(self, position, direction):
        super().__init__(model='sphere', color=color.yellow, scale=0.3,
                         position=position, collider='sphere')
        self.direction = direction
        self.life = 3.0

    def update(self):
        self.position += self.direction * 40 * time.dt
        self.life -= time.dt
        if self.life <= 0:
            destroy(self)


def input(key):
    if key == 'left mouse down':
        shoot_sound.play()        # NEW: play the sound on each shot
        Bullet(position=camera.world_position + camera.forward * 1.5,
               direction=camera.forward)
    if key == 'escape':
        application.quit()

app.run()


## Step 9 — A real gun model

The boxy gun is fine, but we can load a real 3D model. Your project has **`blaster.glb`** (a free CC0 model from Kenney) and its texture in `Textures/`. Swapping it in replaces the three gun cubes with a single line:

```python
gun = Entity(parent=camera, model='blaster.glb',
             position=(0.3, -0.25, 0.5), scale=0.7, rotation=(0, 180, 0))
```

**Troubleshooting to do with the student:** if the barrel points the wrong way, change `rotation` to `(0, 0, 0)`; adjust `scale` / `position` until it looks right.

In [ ]:
import os
# --- WSL / no-GPU fix: use software (CPU) rendering when there's no graphics
#     card, so the 3D window doesn't open black. Harmless on a normal PC.
#     MUST come before importing ursina. ---
if not os.path.exists('/dev/dri'):
    os.environ.setdefault('LIBGL_ALWAYS_SOFTWARE', '1')
    os.environ.setdefault('GALLIUM_DRIVER', 'llvmpipe')

from ursina import *

from ursina.prefabs.first_person_controller import FirstPersonController

app = Ursina()
window.title = 'My First 3D FPS'
window.size = (960, 540)

ground = Entity(model='plane', scale=32, texture='white_cube',
                texture_scale=(32, 32), color=color.green.tint(-0.3),
                collider='box')
Sky()

block_spots = [(5, 5), (5, 8), (5, 11), (-6, 3), (-6, 6), (-9, 6),
               (0, 12), (3, 12), (-3, 12)]
for (x, z) in block_spots:
    Entity(model='cube', position=(x, 1, z), scale=(2, 3, 2),
           texture='brick', color=color.azure, collider='box')

player = FirstPersonController(y=2, origin_y=-0.5, speed=5,
                              mouse_sensitivity=Vec2(20, 20))
player.cursor.enabled = False

crosshair = Entity(parent=camera.ui)
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.03, 0.004))
Entity(parent=crosshair, model='quad', color=color.white, scale=(0.004, 0.03))

# NEW: the real downloaded gun model (replaces the three cubes)
gun = Entity(parent=camera, model='blaster.glb',
             position=(0.3, -0.25, 0.5), scale=0.7, rotation=(0, 180, 0))

shoot_sound = Audio('shoot.wav', volume=0.5, autoplay=False)


class Bullet(Entity):
    def __init__(self, position, direction):
        super().__init__(model='sphere', color=color.yellow, scale=0.3,
                         position=position, collider='sphere')
        self.direction = direction
        self.life = 3.0

    def update(self):
        self.position += self.direction * 40 * time.dt
        self.life -= time.dt
        if self.life <= 0:
            destroy(self)


def input(key):
    if key == 'left mouse down':
        shoot_sound.play()
        Bullet(position=camera.world_position + camera.forward * 1.5,
               direction=camera.forward)
    if key == 'escape':
        application.quit()

app.run()


## Step 10 — Tidy up into a real project

Our single `game.py` works, but it's getting long and everything is jumbled together. Real projects **split the code into files by job**. This is exactly what the finished project in your `fps_3d` folder does:

| File | Its one job |
|------|-------------|
| `config.py` | All the tweakable **numbers** (speed, gun, map, sound) in one place |
| `world.py` | `build_world()` — makes the **ground, blocks, sky** |
| `entities.py` | The **classes**: `Bullet`, `Gun`, `Player`, `Crosshair` |
| `main.py` | The **entry point** — wires it together and runs the game |

Run the finished version with: `python main.py`

### Why split it up? (discuss with your student)
- **Find things fast:** want to change the map? It's in `config.py`. Bullet behaviour? `entities.py`.
- **Each file is short** and does one clear thing.
- **Reuse:** a class like `Bullet` can be used from anywhere, no copy-paste.

### Two nice upgrades the finished project makes
**1. The gun becomes a class — with two interchangeable styles.** A base `Gun` holds the shooting logic; `ModelGun` and `BoxGun` inherit it and only differ in how they *look*. You switch between them with one word in `config.py`:

In [ ]:
# in config.py
GUN_TYPE = "model"   # the downloaded blaster.glb model
# GUN_TYPE = "boxes" # or the gun we build from cubes


```python
# in entities.py (simplified)
class Gun(Entity):
    def shoot(self):                 # written ONCE, used by both guns
        Bullet(position=camera.world_position + camera.forward * 1.5,
               direction=camera.forward)

class ModelGun(Gun):                 # looks like the .glb model
    def __init__(self):
        super().__init__(parent=camera, model='blaster.glb', ...)

class BoxGun(Gun):                   # looks like cubes
    def __init__(self):
        super().__init__(parent=camera, ...)
        # ...build the cubes...
```
Both guns get `shoot()` for free from `Gun`. This idea — different classes that can be used the same way — is called **polymorphism**.

**2. Shooting moves into the `Player` class**, so all the player's controls live in one place (instead of the "magic" top-level `input` function):
```python
class Player(FirstPersonController):
    def __init__(self):
        super().__init__(speed=settings.PLAYER_SPEED, ...)  # walk/look/jump for free
        self.gun = make_gun()
    def input(self, key):
        super().input(key)            # keep the built-in jump etc.
        if key == 'left mouse down':
            self.gun.shoot()
```

> Open `config.py`, `world.py`, `entities.py` and `main.py` in your editor and read them with the student — they'll recognise every piece, because they just built it all by hand.

## Step 11 — Zoom and aim (scroll wheel + right-click)

Let's add scope-style zooming. "Zoom" in 3D means changing the camera's **field of view (FOV)** — how wide an angle the camera sees:

- **smaller FOV → zoomed IN** (narrow view, like a scope)
- **larger FOV → zoomed OUT** (wide view)

We add two controls:
- **Scroll wheel** → step the zoom in/out (and remember it).
- **Hold right-click** → aim (snap to a tight FOV); release → back to the scrolled zoom.

First, add these knobs to **`config.py`**:

In [ ]:
# add to config.py
DEFAULT_FOV = 90      # normal field of view
AIM_FOV = 40          # zoomed-in view while you HOLD right-click
ZOOM_STEP = 5         # how much one scroll-wheel notch changes the zoom
ZOOM_MIN = 30         # most you can zoom IN  with the scroll wheel
ZOOM_MAX = 100        # most you can zoom OUT with the scroll wheel


Then update the **`Player`** class in **`entities.py`**. We keep a `self.base_fov` (the zoom the scroll wheel sets); right-click aiming temporarily overrides it and snaps back on release.

Lines marked `# NEW` are what you add.

In [ ]:
# in entities.py, inside  class Player(FirstPersonController):

    def __init__(self):
        super().__init__(
            y=2, origin_y=-0.5,
            speed=config.PLAYER_SPEED,
            mouse_sensitivity=config.MOUSE_SENSITIVITY,
        )
        self.cursor.enabled = False
        self.gun = make_gun()
        self.base_fov = config.DEFAULT_FOV   # NEW: current zoom level
        camera.fov = self.base_fov           # NEW

    def input(self, key):
        super().input(key)
        if key == 'left mouse down':
            self.gun.shoot()

        # NEW -- scroll wheel zooms in/out
        if key == 'scroll up':
            self.base_fov = max(config.ZOOM_MIN, self.base_fov - config.ZOOM_STEP)
            camera.fov = self.base_fov
        if key == 'scroll down':
            self.base_fov = min(config.ZOOM_MAX, self.base_fov + config.ZOOM_STEP)
            camera.fov = self.base_fov

        # NEW -- hold right-click to aim, release to go back
        if key == 'right mouse down':
            camera.fov = config.AIM_FOV
        if key == 'right mouse up':
            camera.fov = self.base_fov


**What to notice with your student:**
- `camera.fov` is the single thing we change — everything else is just deciding *what value* to set it to.
- `self.base_fov` *remembers* the scrolled zoom, so aiming can return to it.
- `max(...)` / `min(...)` **clamp** the zoom so it can't go too far — the same trick used elsewhere in the engine.

**Controls now:** WASD move, mouse look, Space jump, Left-click shoot, **scroll = zoom**, **hold right-click = aim**, ESC quit.

## Troubleshooting (WSL & common gotchas)

Things we ran into while building this — worth knowing:

- **The window is black** → no graphics card visible to Linux (common on WSL). The software-rendering lines at the top of `main.py` fix it (`LIBGL_ALWAYS_SOFTWARE`).

- **You edited a file but the game still behaves like the old code** → stale compiled bytecode. Python keeps a `__pycache__` folder, and on WSL (editing files from the Windows side) it sometimes runs the *old* `.pyc`. Fix:
  ```bash
  rm -rf __pycache__
  ```
  Or run without caching: `PYTHONDONTWRITEBYTECODE=1 python main.py`

- **Mouse-look doesn't work / cursor feels stuck (WSL only)** → WSL sometimes can't "capture" the mouse (`cannot enable relative mouse mode`). Try restarting WSL (`wsl --shutdown` in Windows PowerShell, then reopen), keep the window on your primary monitor, or run on native Windows where mouse + sound work fully.

- **Don't name your config file `settings.py`** → ursina automatically runs any file named `settings.py` it finds, which causes confusing double-loading. We named ours `config.py` for that reason.

## Recap & where to go next

**What the student learned, in order:**
1. The app / window / run skeleton  
2. Entities (ground, sky, blocks)  
3. **Inheritance** — `FirstPersonController` gave movement for free  
4. Lists + `for` loops to build a map  
5. Screen UI vs. world objects  
6. **Writing a class** (`Bullet`) with its own `update()`  
7. `time.dt` for frame-rate-independent speed  
8. Loading assets once (sound, model)  
9. **Organizing a project** into files + polymorphism

**Challenges to try next:**
- Add **targets** (a `Target` class); make a bullet that hits one destroy it and add to a score.
- Add a **muzzle flash** (a quick light at the barrel on each shot).
- Add **enemies** that slowly move toward the player.
- Add a **HUD** showing score and ammo (using `Text` on `camera.ui`).

Every new feature is just *another entity or another class* — the student already knows how to make both. 